In [3]:
# ==========================================================
# Import Required Libraries
# ==========================================================
!pip install -q -U \
langchain \
langchain-community \
langchain-huggingface \
langchain-text-splitters \
transformers \
sentence-transformers \
faiss-cpu \
accelerate \
pypdf

In [1]:
# ==========================================================
# Import Libraries
# ==========================================================
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("✅ All libraries imported successfully")

/tmp/ipykernel_5203/1808118487.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ All libraries imported successfully


In [2]:
# ==========================================================
# Upload PDF
# ==========================================================
from google.colab import files

uploaded = files.upload()

Saving health-companion-policy-wording.pdf to health-companion-policy-wording.pdf


In [3]:
# ==========================================================
# Load PDF
# ==========================================================
loader = PyPDFLoader("health-companion-policy-wording.pdf")

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 30


In [4]:
# ==========================================================
# Split into Chunks
# ==========================================================
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = text_splitter.split_documents(documents)

print("✅ Total Chunks:", len(chunks))

✅ Total Chunks: 259


In [5]:
# ==========================================================
# Create Embeddings
# ==========================================================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embeddings Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings Loaded


In [6]:
# ==========================================================
# Create FAISS Vector Store
# ==========================================================
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("✅ Vector Store Created")

✅ Vector Store Created


In [7]:
# ==========================================================
# Create Retriever
# ==========================================================
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

print("✅ Retriever Ready")

✅ Retriever Ready


In [8]:
# ==========================================================
# Load FLAN-T5
# ==========================================================
MODEL_NAME = "google/flan-t5-base"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print(f"✅ FLAN-T5 Loaded Successfully on {device}")

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ FLAN-T5 Loaded Successfully on cpu


In [9]:
# ==========================================================
# Create Answer Function
# ==========================================================
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k:v.to(device) for k,v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.3
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [10]:
# ==========================================================
# Ask Questions - complete RAG pipeline
# ==========================================================
question = "What is the waiting period for pre-existing diseases?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

prompt = f"""
You are an insurance assistant.

Answer ONLY using the information below.

Context:

{context}

Question:

{question}

Answer:
"""

answer = generate_answer(prompt)

print("="*80)
print("QUESTION")
print("="*80)

print(question)

print()

print("="*80)
print("ANSWER")
print("="*80)

print(answer)

print()

print("="*80)
print("SOURCE PAGE(S)")
print("="*80)

pages = sorted(
    set(doc.metadata["page"]+1 for doc in docs)
)

print(pages)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION
What is the waiting period for pre-existing diseases?

ANSWER
48 months

SOURCE PAGE(S)
[4, 13]


In [11]:
# ==========================================================
# Interactive Console
# ==========================================================
print("=" * 80)
print("📄 Enterprise Insurance Policy Assistant")
print("Type 'exit' to stop.")
print("=" * 80)

while True:

    question = input("\nAsk a question: ")

    if question.lower() in ["exit", "quit"]:
        print("\nThank you! Exiting the assistant.")
        break

    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are an insurance policy assistant.

Answer ONLY using the information provided below.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generate_answer(prompt)

    print("\n" + "=" * 80)
    print("ANSWER")
    print("=" * 80)
    print(answer)

    pages = sorted(set(doc.metadata["page"] + 1 for doc in docs))

    print("\nSource Page(s):", pages)

📄 Enterprise Insurance Policy Assistant
Type 'exit' to stop.

Ask a question: what is the waiting period for pre-existing disease?

ANSWER
48 months

Source Page(s): [4, 13]

Ask a question: quit

Thank you! Exiting the assistant.
